In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from torch.utils.data import TensorDataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
#GLOBAL

NUM_SAMPLES = 5000
NUM_TIME_STEPS = 1000
T_MAX = 60.0
t_eval = np.linspace(0, T_MAX, NUM_TIME_STEPS)

# parameter ranges, column order: [alpha, beta, delta, gamma, omega, x0, v0]
PARAM_LOW  = np.array([-1.0, 0.1, 0.0, 0.0, 0.5, -2.0, -2.0])
PARAM_HIGH = np.array([ 1.0, 1.0, 0.5, 1.0, 1.5,  2.0,  2.0])

In [ ]:
#GROUND TRUTH

def duffing_deriv(t, state, alpha, beta, delta, gamma, omega):
    x, v = state
    dxdt = v
    dvdt = gamma * np.cos(omega * t) - delta * v - alpha * x - beta * (x**3)
    return [dxdt, dvdt]

In [ ]:
#DATASET GENERATION

def generate_data(num_samples):
    X_data = np.random.uniform(PARAM_LOW, PARAM_HIGH, size=(num_samples, 7))
    Y_data = np.zeros((num_samples, NUM_TIME_STEPS))

    for i in range(num_samples):
        alpha, beta, delta, gamma, omega, x0, v0 = X_data[i]
        params = (alpha, beta, delta, gamma, omega)
        sol = solve_ivp(duffing_deriv, [0, T_MAX], [x0, v0], args=params, t_eval=t_eval, rtol=1e-8, atol=1e-10)
        Y_data[i] = sol.y[0]

    return torch.tensor(X_data, dtype=torch.float32), torch.tensor(Y_data, dtype=torch.float32)

In [ ]:
#NEURAL NETWORK

class DuffingNet(nn.Module):
    def __init__(self, input_dim=8, output_dim=1):
        super(DuffingNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.GELU(),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        # x: (N, 8) = [alpha, beta, delta, gamma, omega, x0, v0, t]
        return self.net(x)


# time grid fed to the network, normalized to [0, 1] so t has a similar scale to the other inputs
t_grid = torch.tensor(t_eval / T_MAX, dtype=torch.float32, device=device)

def predict_trajectory(model, params, t=t_grid):
    """Evaluate the pointwise model on every time in t for each parameter set.
    params: (B, 7), t: (T,)  ->  returns (B, T)"""
    B, T = params.shape[0], t.shape[0]
    p  = params.unsqueeze(1).expand(B, T, params.shape[1])   # (B, T, 7)
    tt = t.view(1, T, 1).expand(B, T, 1)                      # (B, T, 1)
    inp = torch.cat([p, tt], dim=-1).reshape(B * T, -1)       # (B*T, 8)
    return model(inp).view(B, T)

In [ ]:
#HYBRID LOSS

XI = 0.65           #weight balancing
N_COL = 2048        # random (params, t) collocation points per step
W_IC = 1.0          # weight of the initial-condition term inside the physics loss
PHYS_SCALE = 1.0    # adjust if losses are orders of magnitude different

p_low  = torch.tensor(PARAM_LOW,  dtype=torch.float32, device=device)
p_high = torch.tensor(PARAM_HIGH, dtype=torch.float32, device=device)

def sample_params(n):
    return p_low + (p_high - p_low) * torch.rand(n, 7, device=device)

def physics_loss_fn(model, n=N_COL):
    # enable_grad so this also works inside the torch.no_grad() test loop
    with torch.enable_grad():
        params = sample_params(n)
        alpha, beta, delta, gamma, omega, x0, v0 = params.split(1, dim=1)   # each (n, 1)

        # ODE residual at random times (t in seconds; the network sees t / T_MAX)
        t = (T_MAX * torch.rand(n, 1, device=device)).requires_grad_(True)
        x    = model(torch.cat([params, t / T_MAX], dim=1))
        x_t  = torch.autograd.grad(x,   t, torch.ones_like(x),   create_graph=True)[0]
        x_tt = torch.autograd.grad(x_t, t, torch.ones_like(x_t), create_graph=True)[0]

        residual = x_tt + delta * x_t + alpha * x + beta * x**3 \
                   - gamma * torch.cos(omega * t)
        res_loss = torch.mean(residual**2)

        # initial conditions: x(0) = x0, dx/dt(0) = v0
        t0  = torch.zeros(n, 1, device=device, requires_grad=True)
        x_0 = model(torch.cat([params, t0 / T_MAX], dim=1))
        v_0 = torch.autograd.grad(x_0, t0, torch.ones_like(x_0), create_graph=True)[0]
        ic_loss = torch.mean((x_0 - x0)**2 + (v_0 - v0)**2)

    return (res_loss + W_IC * ic_loss) / PHYS_SCALE

def compute_loss(model, batch_x, batch_y, xi=XI):
    pred = predict_trajectory(model, batch_x)      # (B, 100) on the data grid
    truth_loss   = criterion(pred, batch_y)
    physics_loss = physics_loss_fn(model)          # independent random points
    total_loss   = xi * physics_loss + (1 - xi) * truth_loss
    return total_loss, truth_loss, physics_loss

In [ ]:
#training loop
X, Y = generate_data(NUM_SAMPLES)
dataset = TensorDataset(X, Y)

train_size = 4000
test_size = 1000
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

model = DuffingNet().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 500
for epoch in range(epochs):
    model.train()
    train_loss = train_truth = train_phys = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        optimizer.zero_grad()
        loss, t_l, p_l = compute_loss(model, batch_x, batch_y)
        loss.backward()
        optimizer.step()
        train_loss  += loss.item()
        train_truth += t_l.item()
        train_phys  += p_l.item()

    model.eval()
    test_loss = test_truth = test_phys = 0.0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            loss, t_l, p_l = compute_loss(model, batch_x, batch_y)
            test_loss  += loss.item()
            test_truth += t_l.item()
            test_phys  += p_l.item()

    if (epoch + 1) % 10 == 0:
        n_tr, n_te = len(train_loader), len(test_loader)
        m = evaluate_metrics(model, test_loader)
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train total: {train_loss/n_tr:.4f} (truth {train_truth/n_tr:.4f}, phys {train_phys/n_tr:.4f}) | "
              f"Test total: {test_loss/n_te:.4f} (truth {test_truth/n_te:.4f}, phys {test_phys/n_te:.4f})")
        print(f"    rel L2: mean {m['rel_l2_mean']:.2%}, median {m['rel_l2_median']:.2%} | "
              f"within {TOL}: {m['tol_acc']:.2%}")

In [ ]:
#METRICS

TOLERANCE = 0.05   # absolute position tolerance for "within tolerance" accuracy

def evaluate_metrics(model, loader, tol=TOLERANCE, eps=1e-8):
    model.eval()
    rel_l2_all, hits, total = [], 0, 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            pred = predict_trajectory(model, batch_x)                # (B, NUM_TIME_STEPS)

            # relative L2: one value per trajectory
            rel_l2 = torch.norm(pred - batch_y, dim=1) / (torch.norm(batch_y, dim=1) + eps)
            rel_l2_all.append(rel_l2)

            # within tolerance: count points closer than tol
            hits  += (torch.abs(pred - batch_y) < tol).sum().item()
            total += batch_y.numel()

    rel_l2_all = torch.cat(rel_l2_all)
    return {
        "rel_l2_mean":   rel_l2_all.mean().item(),
        "rel_l2_median": rel_l2_all.median().item(),
        "tol_acc":       hits / total,
    }